In [12]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [13]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    BASE_DIR = cwd / "notebooks" if (cwd / "notebooks").exists() else cwd

DATASET_PATH = str((BASE_DIR / ".." / "projeto" / "dados" / "dataset_tratado_final.csv").resolve())
OUTPUT_PATH = str(BASE_DIR / "eda.png")

PALETTE = {
    "primary":  "#1B4F72",
    "accent":   "#E74C3C",
    "success":  "#27AE60",
    "warn":     "#F39C12",
    "light":    "#AED6F1",
    "grid":     "#EAECEE",
    "bg":       "#FDFEFE",
}

COUNTRY_COLORS = {
    "USA":          "#1B4F72",
    "AUSTRALIA":    "#E74C3C",
    "SOUTH AFRICA": "#27AE60",
    "BRAZIL":       "#F39C12",
    "MEXICO":       "#8E44AD",
}

In [14]:
# ---------------------------------------------------------------------------
# 1. CARREGAMENTO E TRATAMENTO DOS DADOS
# ---------------------------------------------------------------------------

def load_and_clean() -> pd.DataFrame:
    df = pd.read_csv(DATASET_PATH, encoding="latin-1")

    # Renomear colunas com encoding problemático
    rename = {}
    for col in df.columns:
        if "vel da mar" in col.lower() or "nivel" in col.lower():
            rename[col] = "Nível da maré"
        elif "temperatura do mar" in col.lower():
            rename[col] = "Temperatura do mar (°C)"
    df.rename(columns=rename, inplace=True)

    # Remover ano claramente errado (3019)
    df = df[df["Year"] <= 2025].copy()

    # Fatal: normalizar para booleano
    df["fatal"] = df["Fatal (Y/N)"].str.strip().str.upper() == "Y"

    # Age: converter para numérico (há entradas textuais)
    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

    return df

In [15]:
# ---------------------------------------------------------------------------
# 2. ESTATÍSTICAS DESCRITIVAS
# ---------------------------------------------------------------------------

def print_descriptive_stats(df: pd.DataFrame):
    print("=" * 60)
    print("ESTATÍSTICAS DESCRITIVAS — DeepSearch EDA")
    print("=" * 60)
    print(f"Total de registros : {len(df)}")
    print(f"Total de colunas   : {len(df.columns)}")
    print(f"Período coberto    : {int(df['Year'].min())} – {int(df['Year'].max())}")
    print(f"Países             : {df['Country'].nunique()} ({', '.join(df['Country'].unique())})")
    print()

    print("--- Valores nulos (colunas relevantes) ---")
    relevant = ["Date", "Year", "Country", "Activity", "Fatal (Y/N)",
                "Age", "Time", "Injury_category",
                "Temperatura do mar (°C)", "temp_media_C",
                "precipitacao_mm", "vento_max_kmh"]
    nulls = df[relevant].isnull().sum()
    for col, n in nulls.items():
        pct = n / len(df) * 100
        print(f"  {col:<30}: {n:>4} nulos ({pct:.1f}%)")

    print()
    print("--- Estatísticas das variáveis ambientais numéricas ---")
    env_cols = ["Temperatura do mar (°C)", "temp_media_C",
                "precipitacao_mm", "vento_max_kmh", "umidade_max_pct"]
    print(df[env_cols].describe().round(2).to_string())
    print()

    print("--- Distribuição por país ---")
    print(df["Country"].value_counts().to_string())
    print()

    print("--- Distribuição por atividade (top 10) ---")
    print(df["Activity"].value_counts().head(10).to_string())
    print()

    print("--- Distribuição por gravidade ---")
    print(df["Injury_category"].value_counts().to_string())
    print("=" * 60)


In [16]:
# ---------------------------------------------------------------------------
# 3. HELPERS DE ESTILO
# ---------------------------------------------------------------------------

def style_ax(ax, title, xlabel="", ylabel=""):
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8, color="#2C3E50")
    ax.set_xlabel(xlabel, fontsize=9, color="#5D6D7E")
    ax.set_ylabel(ylabel, fontsize=9, color="#5D6D7E")
    ax.set_facecolor(PALETTE["bg"])
    ax.grid(True, color=PALETTE["grid"], linewidth=0.7, linestyle="--", alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8, colors="#5D6D7E")


def add_note(ax, text):
    ax.text(
        0.01, -0.22, text,
        transform=ax.transAxes,
        fontsize=7.5, color="#5D6D7E",
        verticalalignment="top",
        wrap=True,
    )

In [17]:
# ---------------------------------------------------------------------------
# 4. VISUALIZAÇÕES
# ---------------------------------------------------------------------------

def plot_incidentes_por_pais(ax, df):
    """1. Distribuição absoluta de incidentes por país."""
    counts = df["Country"].value_counts()
    colors = [COUNTRY_COLORS.get(c, PALETTE["light"]) for c in counts.index]

    bars = ax.barh(counts.index[::-1], counts.values[::-1],
                   color=colors[::-1], edgecolor="white", height=0.6)

    for bar, val in zip(bars, counts.values[::-1]):
        ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
                f"{val}", va="center", fontsize=8.5, color="#2C3E50", fontweight="bold")

    ax.set_xlim(0, counts.max() * 1.15)
    style_ax(ax, "1. Incidentes por País", "Número de incidentes", "")
    add_note(ax, "EUA concentra 61% dos registros, seguido por Austrália (21%).\n"
                 "O Brasil representa apenas 4%, mas é relevante por ser o foco do sistema.")


def plot_evolucao_temporal(ax, df):
    """2. Evolução anual de incidentes (1980–2020)."""
    df_ano = df[(df["Year"] >= 1980) & (df["Year"] <= 2020)]
    por_ano = df_ano.groupby("Year").size().reset_index(name="count")

    ax.fill_between(por_ano["Year"], por_ano["count"],
                    alpha=0.18, color=PALETTE["primary"])
    ax.plot(por_ano["Year"], por_ano["count"],
            color=PALETTE["primary"], linewidth=2)

    # Destaque do pico
    pico = por_ano.loc[por_ano["count"].idxmax()]
    ax.scatter(pico["Year"], pico["count"],
               color=PALETTE["accent"], s=60, zorder=5)
    ax.annotate(f"Pico: {int(pico['Year'])} ({int(pico['count'])})",
                xy=(pico["Year"], pico["count"]),
                xytext=(pico["Year"] - 8, pico["count"] + 5),
                fontsize=7.5, color=PALETTE["accent"],
                arrowprops=dict(arrowstyle="->", color=PALETTE["accent"], lw=1))

    style_ax(ax, "2. Evolução Anual de Incidentes (1980–2020)",
             "Ano", "Número de incidentes")
    add_note(ax, "Tendência de crescimento de 1980 a 2015, com pico em 2015.\n"
                 "Queda em 2020 explicada pela pandemia (dados incompletos no período).")


def plot_atividades(ax, df):
    """3. Top 10 atividades com maior número de incidentes."""
    top = df["Activity"].value_counts().head(10)
    norm = top.values / top.values.max()
    colors = plt.cm.Blues(0.35 + 0.55 * norm)

    bars = ax.barh(top.index[::-1], top.values[::-1],
                   color=colors[::-1], edgecolor="white", height=0.65)

    for bar, val in zip(bars, top.values[::-1]):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                f"{val}", va="center", fontsize=8, color="#2C3E50")

    ax.set_xlim(0, top.max() * 1.15)
    style_ax(ax, "3. Atividades Mais Envolvidas em Ataques",
             "Número de incidentes", "")
    add_note(ax, "Surf lidera com 40% dos incidentes, coerente com maior exposição e\n"
                 "tempo passado na zona de arrebentação, área de caça de tubarões.")


def plot_gravidade(ax, df):
    """4. Gravidade dos ferimentos (Injury Category)."""
    order = ["Fatal", "Amputation / Severe", "Laceration",
             "Bite", "Minor Injury", "No Injury", "Unknown"]
    counts = df["Injury_category"].value_counts().reindex(order).dropna()

    pal = [PALETTE["accent"], "#C0392B", PALETTE["warn"],
           "#F8C471", PALETTE["success"], PALETTE["light"], "#BDC3C7"]

    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=None,
        colors=pal[:len(counts)],
        autopct="%1.1f%%",
        startangle=140,
        pctdistance=0.78,
        wedgeprops=dict(edgecolor="white", linewidth=1.2),
    )
    for at in autotexts:
        at.set_fontsize(7.5)

    ax.legend(counts.index, loc="lower left",
              fontsize=7, framealpha=0.8,
              bbox_to_anchor=(-0.05, -0.05))

    ax.set_title("4. Gravidade dos Ferimentos", fontsize=11,
                 fontweight="bold", pad=8, color="#2C3E50")
    add_note(ax, "Laceração (37%) e mordida (22%) dominam. Apenas 8,5% dos ataques\n"
                 "são fatais, indicando que a maioria é de baixa ou média gravidade.")


def plot_temperatura_por_pais(ax, df):
    """5. Distribuição da temperatura do mar por país (boxplot)."""
    col = "Temperatura do mar (°C)"
    df_clean = df[[col, "Country"]].dropna()

    order = df_clean.groupby("Country")[col].median().sort_values(ascending=False).index
    colors = [COUNTRY_COLORS.get(c, PALETTE["light"]) for c in order]

    bp = ax.boxplot(
        [df_clean[df_clean["Country"] == c][col].values for c in order],
        vert=True,
        patch_artist=True,
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker="o", markersize=3, alpha=0.4),
        widths=0.55,
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)

    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels([c.title() for c in order], fontsize=8, rotation=10)
    style_ax(ax, "5. Temperatura do Mar por País (°C)",
             "", "Temperatura (°C)")
    add_note(ax, "Brasil e México apresentam as temperaturas mais altas (>25°C),\n"
                 "favorecendo maior atividade de tubarões em águas quentes.")


def plot_correlacao(ax, df):
    """6. Heatmap de correlação entre variáveis ambientais."""
    cols = {
        "Temperatura do mar (°C)": "Temp. Mar",
        "temp_media_C":            "Temp. Ar",
        "precipitacao_mm":         "Precipit.",
        "vento_max_kmh":           "Vento",
        "umidade_max_pct":         "Umidade",
    }
    sub = df[list(cols.keys())].dropna()
    sub.columns = list(cols.values())
    corr = sub.corr()

    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1, vmax=1,
        linewidths=0.5,
        linecolor="white",
        annot_kws={"size": 8},
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title("6. Correlação entre Variáveis Ambientais",
                 fontsize=11, fontweight="bold", pad=8, color="#2C3E50")
    ax.tick_params(labelsize=8)
    add_note(ax, "Forte correlação (0,77) entre temperatura do mar e temperatura do ar.\n"
                 "Vento correlaciona negativamente com umidade (-0,24): dias secos tendem a ser mais ventosos.")


def plot_fatal_por_pais(ax, df):
    """7. Proporção de incidentes fatais vs não fatais por país."""
    df_f = df[df["Fatal (Y/N)"].isin(["Y", "N"])].copy()
    df_f["fatal_label"] = df_f["Fatal (Y/N)"].map({"Y": "Fatal", "N": "Não fatal"})

    pivot = df_f.groupby(["Country", "fatal_label"]).size().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    pivot_pct = pivot_pct.reindex(
        df_f["Country"].value_counts().index
    )

    pivot_pct[["Não fatal", "Fatal"]].plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[PALETTE["light"], PALETTE["accent"]],
        edgecolor="white",
        width=0.55,
    )

    ax.set_xticklabels([c.title() for c in pivot_pct.index],
                       rotation=15, fontsize=8)
    ax.set_ylim(0, 108)
    ax.legend(["Não fatal", "Fatal"], fontsize=8, framealpha=0.8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    style_ax(ax, "7. Proporção Fatal vs Não Fatal por País",
             "", "% dos incidentes")
    add_note(ax, "Austrália apresenta a maior taxa de fatalidade (~12%), seguida por\n"
                 "África do Sul. EUA tem a menor proporção apesar do maior volume.")


In [18]:
# ---------------------------------------------------------------------------
# 5. FIGURA PRINCIPAL
# ---------------------------------------------------------------------------

def main():
    print("Carregando e limpando dados...")
    df = load_and_clean()

    print_descriptive_stats(df)

    print("Gerando visualizações...")
    fig = plt.figure(figsize=(20, 22))
    fig.patch.set_facecolor(PALETTE["bg"])

    fig.suptitle(
        "Análise Exploratória de Dados — Ataques de Tubarão\n"
        "DeepSearch · CESAR School · Machine Learning I 2026.1",
        fontsize=15, fontweight="bold", color="#1B2631", y=0.99,
    )

    gs = gridspec.GridSpec(
        4, 2, figure=fig,
        hspace=0.72, wspace=0.35,
        left=0.07, right=0.96, top=0.96, bottom=0.04,
    )

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    ax5 = fig.add_subplot(gs[2, 0])
    ax6 = fig.add_subplot(gs[2, 1])
    ax7 = fig.add_subplot(gs[3, :])

    plot_incidentes_por_pais(ax1, df)
    plot_evolucao_temporal(ax2, df)
    plot_atividades(ax3, df)
    plot_gravidade(ax4, df)
    plot_temperatura_por_pais(ax5, df)
    plot_correlacao(ax6, df)
    plot_fatal_por_pais(ax7, df)

    fig.savefig(OUTPUT_PATH, dpi=150, bbox_inches="tight",
                facecolor=PALETTE["bg"])
    print(f"\nFigura salva em: {OUTPUT_PATH}")


In [19]:
main()
plt.show()

Carregando e limpando dados...
ESTATÍSTICAS DESCRITIVAS — DeepSearch EDA
Total de registros : 2559
Total de colunas   : 41
Período coberto    : 1980 – 2020
Países             : 5 (USA, AUSTRALIA, SOUTH AFRICA, MEXICO, BRAZIL)

--- Valores nulos (colunas relevantes) ---
  Date                          :  160 nulos (6.3%)
  Year                          :    0 nulos (0.0%)
  Country                       :    0 nulos (0.0%)
  Activity                      :    0 nulos (0.0%)
  Fatal (Y/N)                   :  188 nulos (7.3%)
  Age                           :  665 nulos (26.0%)
  Time                          :  728 nulos (28.4%)
  Injury_category               :    0 nulos (0.0%)
  Temperatura do mar (°C)       :    0 nulos (0.0%)
  temp_media_C                  :  289 nulos (11.3%)
  precipitacao_mm               :  289 nulos (11.3%)
  vento_max_kmh                 :  289 nulos (11.3%)

--- Estatísticas das variáveis ambientais numéricas ---
       Temperatura do mar (°C)  temp_media_C